# M2 RQ1/RQ2 · Tesla T4
결과 ZIP 및 입력 ZIP은 사용자 Drive에서 읽고 출력만 새 실행 디렉터리에 저장한다. 반복 실행은 새 런타임에서 시작한다.

In [ ]:
import os, sys, subprocess, json, hashlib, tempfile, zipfile
from pathlib import Path
from datetime import datetime, timezone
from google.colab import auth, drive
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import torch
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(), 'Tesla T4 required'
work=Path(tempfile.mkdtemp(dir='/content'))
repo=work/'repo'
def run(*args):
    subprocess.run(args,check=True)
run('git','clone','--quiet','--branch','experiment/m2-selective-refinement-20261121','https://github.com/darkkyoung/REWIND-like-prototype.git',str(repo))
os.chdir(repo)
run('git','checkout','--detach','a4cd971072e8537c61e2e4337d87931f114bef88')
sys.path.insert(0,str(repo))
drive.mount('/content/drive')
auth.authenticate_user()
import google.auth
creds,_=google.auth.default()
service=build('drive','v3',credentials=creds)
archive_id={}
handoff=Path('/content/drive/MyDrive/rewind_selective')/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
handoff.mkdir(parents=True,exist_ok=False)

In [ ]:
def download(fid,target,expected_size,expected_sha=None):
    meta=service.files().get(fileId=fid,fields='id,name,size,sha256Checksum,version').execute()
    if int(meta['size'])!=expected_size:
        raise ValueError(meta['name'])
    with target.open('xb') as stream:
        reader=MediaIoBaseDownload(stream,service.files().get_media(fileId=fid))
        done=False
        while not done:
            _,done=reader.next_chunk(num_retries=3)
    digest=hashlib.sha256(target.read_bytes()).hexdigest()
    if digest!=meta['sha256Checksum'] or (expected_sha and digest!=expected_sha):
        raise ValueError('archive SHA mismatch')
    archive_id[fid]=meta
download('1V11xsktR_TNybTYXDD6RDF45ZgjiVnqZ',work/'inputs.zip',2867728,'d786d36034f7c430d7eb7d3aa6bbee1a26b08308f7f75162623bceb67cec8082')
download('1MBVYoI0L77N4fc0ZQfrLKiFfKFNyz5AC',work/'result.zip',42297177)
(handoff/'archive_identity.json').write_text(json.dumps(archive_id,indent=2))
with zipfile.ZipFile(work/'inputs.zip') as z:
    z.extractall(work/'input_bundle')
from experiments.final_m2.selective_refinement.bundle import unpack_results
unpack_results(work/'result.zip',work/'frozen')

In [ ]:
run(sys.executable,'-m','unittest','experiments.final_m2.selective_refinement.test_phase','-v')
run(sys.executable,'-m','unittest','discover','-s','tests','-p','test_m2.py','-v')
run(sys.executable,'-m','experiments.final_m2.bone_length_loss_20261117.prepare_candidate',work/'input_bundle/poses',work/'candidate')
run(sys.executable,'-m','experiments.final_m2.selective_refinement.runner','--inputs',work/'input_bundle/inputs','--candidate',work/'candidate/joyful57.npz','--results',work/'frozen','--out',handoff/'results')
print((handoff/'results/summary.json').read_text())